# Driver Drowsiness Detection - ANN Model

In [ ]:
# STEP 1 - IMPORT LIBRARIES
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

IMG_SIZE = 64
print('All Libraries Imported Successfully')
print('TensorFlow Version:', tf.__version__)

In [ ]:
# STEP 2 - DEFINE DATASET PATHS AND LABELS
source_folders = {
    'eyes/train/Close' : r'C:\Users\semwa\Project_Bnat\archive (1)\data\eyes\train\Close',
    'eyes/train/Open'  : r'C:\Users\semwa\Project_Bnat\archive (1)\data\eyes\train\Open',
    'eyes/val/Close'   : r'C:\Users\semwa\Project_Bnat\archive (1)\data\eyes\val\Close',
    'eyes/val/Open'    : r'C:\Users\semwa\Project_Bnat\archive (1)\data\eyes\val\Open',
    'eyes/test/Close'  : r'C:\Users\semwa\Project_Bnat\archive (1)\data\eyes\test\Close',
    'eyes/test/Open'   : r'C:\Users\semwa\Project_Bnat\archive (1)\data\eyes\test\Open',
    'yawn/yawn'        : r'C:\Users\semwa\Project_Bnat\archive (1)\data\yawn\yawn',
    'yawn/no_yawn'     : r'C:\Users\semwa\Project_Bnat\archive (1)\data\yawn\no yawn',
}

# 1 = Drowsy (Close eyes / Yawning)
# 0 = Alert  (Open eyes  / No Yawn)
label_map = {
    'eyes/train/Close' : 1,
    'eyes/train/Open'  : 0,
    'eyes/val/Close'   : 1,
    'eyes/val/Open'    : 0,
    'eyes/test/Close'  : 1,
    'eyes/test/Open'   : 0,
    'yawn/yawn'        : 1,
    'yawn/no_yawn'     : 0,
}

print('Paths and Labels Defined')
for key, path in source_folders.items():
    exists = os.path.isdir(path)
    print(f'{key} --> EXISTS: {exists}')

In [ ]:
# STEP 3 - LOAD IMAGES AND LABELS
X = []
y = []

for key, path in source_folders.items():
    label = label_map[key]
    print(f'Loading: {key} ...')
    count = 0
    for file in os.listdir(path):
        file_path = os.path.join(path, file)
        img = cv2.imread(file_path)
        if img is None:
            continue
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        X.append(img)
        y.append(label)
        count += 1
    print(f'  Loaded {count} images')

X = np.array(X)
y = np.array(y)

print(f'\nTotal Images : {X.shape[0]}')
print(f'Image Shape  : {X.shape}')
print(f'Labels Shape : {y.shape}')
print(f'Drowsy (1)   : {np.sum(y == 1)}')
print(f'Alert  (0)   : {np.sum(y == 0)}')

In [ ]:
# STEP 4 - NORMALIZE AND FLATTEN

# Normalize pixel values from 0-255 to 0-1
X = X / 255.0

# Flatten 64x64 image into 4096 values (ANN needs 1D input)
X = X.reshape(X.shape[0], -1)

print(f'After Normalize and Flatten : {X.shape}')
print(f'Each image is now a vector of {X.shape[1]} values')

In [ ]:
# STEP 5 - TRAIN / VAL / TEST SPLIT
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f'Train : {X_train.shape[0]} images')
print(f'Val   : {X_val.shape[0]} images')
print(f'Test  : {X_test.shape[0]} images')

In [ ]:
# STEP 6 - VISUALIZE SAMPLE IMAGES
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sample Images (0=Alert, 1=Drowsy)', fontsize=14)

for i in range(10):
    ax = axes[i // 5][i % 5]
    img = X_train[i].reshape(IMG_SIZE, IMG_SIZE)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# STEP 7 - BUILD THE ANN MODEL
model = Sequential([
    Dense(512, activation='relu', input_shape=(IMG_SIZE * IMG_SIZE,)),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64,  activation='relu'),
    Dense(1,   activation='sigmoid')  # binary: drowsy or alert
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# STEP 8 - TRAIN THE MODEL
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=1
)

print('Model Training Completed')

In [ ]:
# STEP 9 - PLOT TRAINING ACCURACY AND LOSS
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# STEP 10 - EVALUATE ON TEST DATA
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc * 100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')

In [ ]:
# STEP 11 - CLASSIFICATION REPORT AND CONFUSION MATRIX
y_pred = (model.predict(X_test) > 0.5).astype(int)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Alert (0)', 'Drowsy (1)']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Alert', 'Drowsy'],
            yticklabels=['Alert', 'Drowsy'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# STEP 12 - SAVE THE MODEL
save_path = r'C:\Users\semwa\Project_Bnat\drowsiness_ann_model.h5'
model.save(save_path)
print(f'Model Saved Successfully at: {save_path}')